In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

# --- 0. STYLE SETTINGS ---
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12
})

# --- 1. DATA LOADING ---
ins_df = pd.read_csv('medical_insurance.csv')
dia_df = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')

# --- 2. MULTI-PLOT CANVAS SETUP (2 across, 3 down) ---
fig, axes = plt.subplots(3, 2, figsize=(14, 18))
axes = axes.flatten()

# Figure 1: Target Skewness
sns.histplot(ins_df['annual_medical_cost'], kde=True, color='#1c4e80', ax=axes[0], bins=40)
axes[0].set_title('Figure 1: Right-Skewed Annual Medical Costs (Target)', fontweight='bold')
axes[0].set_xlabel('Annual Cost ($)')

# Figure 2: Correlations
corr_cols = ['age', 'bmi', 'income', 'risk_score', 'annual_medical_cost', 'monthly_premium']
sns.heatmap(ins_df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", cbar=False, ax=axes[1])
axes[1].set_title('Figure 2: Multivariate Actuarial Correlations', fontweight='bold')

# Figure 3: Non-Linear Interaction Surface
sns.scatterplot(data=ins_df.sample(n=1000, random_state=42), x='bmi', y='annual_medical_cost', hue='smoker', alpha=0.6, ax=axes[2])
axes[2].set_title('Figure 3: W1 Non-Linear Interaction: BMI × Smoker', fontweight='bold')

# Figure 4: Week 4 Logistic Regression Depth Focus
features_lr = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'Age', 'Income']
X_lr = dia_df[features_lr].dropna()
y_lr_binary = np.where(dia_df.loc[X_lr.index, 'Diabetes_012'] > 0, 1, 0)
X_lr_scaled = StandardScaler().fit_transform(X_lr)
lr_model = LogisticRegression().fit(X_lr_scaled, y_lr_binary)
pd.Series(lr_model.coef_[0], index=features_lr).sort_values().plot(kind='barh', color='#00a896', ax=axes[3])
axes[3].axvline(0, color='black', linestyle='--')
axes[3].set_title('Figure 4: W4 Log-Odds Coefficients (Scaled Profile)', fontweight='bold')

# Figure 5: Week 3 PCA Space
X_pca = dia_df[features_lr].dropna().sample(n=2000, random_state=42)
pcs = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_pca))
scatter = axes[4].scatter(pcs[:, 0], pcs[:, 1], c=dia_df.loc[X_pca.index, 'Diabetes_012'], cmap='viridis', alpha=0.5, s=15)
axes[4].set_title('Figure 5: W3 PCA Latent Space Separation', fontweight='bold')

# Figure 6: Week 6 Random Forest Depth Focus
features_rf = ['age', 'bmi', 'income', 'risk_score', 'visits_last_year', 'household_size']
X_rf = ins_df[features_rf].fillna(ins_df[features_rf].median())
rf_model = RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1).fit(X_rf, ins_df['annual_medical_cost'])
pd.Series(rf_model.feature_importances_, index=features_rf).sort_values().plot(kind='barh', color='#1c4e80', ax=axes[5])
axes[5].set_title('Figure 6: W6 Random Forest Pure Actuarial Importance', fontweight='bold')

plt.tight_layout()
plt.savefig('milestone_one_grid.png', dpi=300, bbox_inches='tight')
plt.show()

ModuleNotFoundError: No module named 'seaborn'